In [ ]:
# ==============================
# CREDIT RISK ASSESSMENT PROJECT
# ==============================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor


# ==============================
# 2. UPLOAD DATASET
# ==============================
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])

print("\nDataset Loaded Successfully!\n")
print(df.head())
print(df.info())


# ==============================
# 3. DATA PREPROCESSING
# ==============================

# Handle Missing Values
df = df.dropna()

# Encode Categorical Columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

print("\nAfter Encoding:\n", df.head())


# ==============================
# 4. DEFINE FEATURES & TARGET
# ==============================

# ✅ Correct target column (as per your dataset)
target_column = 'loan_int_rate'

X = df.drop(columns=[target_column])
y = df[target_column]


# ==============================
# 5. FEATURE SCALING
# ==============================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ==============================
# 6. TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


# ==============================
# 7. MODEL TRAINING
# ==============================

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# 2. OLS Regression
X_ols = sm.add_constant(X_scaled)
model_ols = sm.OLS(y, X_ols).fit()

# 3. SVR
svr = SVR(kernel='rbf')
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

# 4. Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# 5. Neural Network
mlp = MLPRegressor(hidden_layer_sizes=(100,50), max_iter=500)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)


# ==============================
# 8. EVALUATION FUNCTION
# ==============================

def evaluate(y_test, y_pred, model_name):
    print(f"\n===== {model_name} =====")
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R2 Score:", r2_score(y_test, y_pred))


# Evaluate All Models
evaluate(y_test, y_pred_lr, "Linear Regression")
evaluate(y_test, y_pred_svr, "SVR")
evaluate(y_test, y_pred_dt, "Decision Tree")
evaluate(y_test, y_pred_mlp, "Neural Network")


# ==============================
# 9. OLS SUMMARY (STATISTICS)
# ==============================

print("\n===== OLS SUMMARY =====")
print(model_ols.summary())


# ==============================
# 10. VIF CALCULATION
# ==============================

vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns

vif_data["VIF"] = [
    variance_inflation_factor(X_scaled, i)
    for i in range(X_scaled.shape[1])
]

print("\n===== VIF VALUES =====")
print(vif_data)


# ==============================
# 11. MODEL COMPARISON
# ==============================

results = pd.DataFrame({
    "Model": ["Linear", "SVR", "Decision Tree", "Neural Network"],
    "R2 Score": [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_svr),
        r2_score(y_test, y_pred_dt),
        r2_score(y_test, y_pred_mlp)
    ]
})

print("\n===== MODEL COMPARISON =====")
print(results)


# ==============================
# 12. FEATURE IMPACT (IMPORTANT)
# ==============================

print("\n===== FEATURE COEFFICIENTS (Linear Regression) =====")
coeff_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_
})

print(coeff_df)


# ✅ Salary → person_income
if 'person_income' in X.columns:
    print("\nImpact of Income on Interest Rate:")
    print(coeff_df[coeff_df['Feature'] == 'person_income'])

# ✅ Loan Amount → loan_amnt
if 'loan_amnt' in X.columns:
    print("\nImpact of Loan Amount on Interest Rate:")
    print(coeff_df[coeff_df['Feature'] == 'loan_amnt'])

Saving credit_risk_dataset.csv to credit_risk_dataset.csv

Dataset Loaded Successfully!

   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist